In [ ]:
# Quantum simulation libraries
from qutip import (
    basis, 
    mesolve, 
    qeye, 
    sigmax, 
    sigmay, 
    sigmaz, 
    tensor,
    )
import qutip

# Machine learning libraries
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

# Plotting libraries
import matplotlib.pyplot as plt
from rich.progress import track
import seaborn as sns

# Linalg libraries
import numpy as np
from scipy.stats import pearsonr
from scipy.integrate import odeint

# Helper libraries
from dataclasses import dataclass


In [ ]:
# set the device
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using {device} device")

# Load target data

In [ ]:
def rk4_step(f, x, t, dt, *args):
    """Performs a single RK4 step."""
    k1 = dt * f(x, t, *args)
    k2 = dt * f(x + 0.5 * k1, t + 0.5 * dt, *args)
    k3 = dt * f(x + 0.5 * k2, t + 0.5 * dt, *args)
    k4 = dt * f(x + k3, t + dt, *args)
    return x + (k1 + 2*k2 + 2*k3 + k4) / 6

def mackey_glass_eq(x_t, t, beta, gamma, tau, n, history, dt):
    """Mackey-Glass equation with delay."""
    # Linear interpolation for delay
    if t <= tau:
        x_delay = history[0]
    else:
        t_index = int(t / dt)
        tau_index = int(tau / dt)
        x_delay = history[max(t_index - tau_index, 0)]
    
    dxdt = beta * x_delay / (1 + x_delay ** n) - gamma * x_t
    return dxdt

# Parameters
beta = 0.2
gamma = 0.1
tau = 17
n = 10
dt = 0.1
N = 5000  # Total number of steps

# Initial conditions
x0 = 0.5

# Prepare the history array with initial condition
history = np.array([x0])

# Time loop
for i in range(1, N + 1):
    t = i * dt
    x_t = history[-1]
    x_next = rk4_step(mackey_glass_eq, x_t, t, dt, beta, gamma, tau, n, history, dt)
    history = np.append(history, x_next)

# Plotting
t = np.arange(N + 1) * dt
plt.figure(figsize=(10, 6))
plt.plot(t, history, label='Mackey-Glass Equation')
plt.xlabel('Time')
plt.ylabel('x(t)')
plt.title('Mackey-Glass Equation Solution with JAX and RK4')
plt.legend()
plt.show()

In [ ]:
sampled_data = history[::20]

In [ ]:
plt.plot(sampled_data, '.')

In [ ]:
class BFieldGenerator:
    def __init__(
        self, 
        data: np.ndarray,
        update_rate: int,
        times: np.ndarray
    ):
        """
        Create the BField generator for the Lorenz Oscillator.
        """
        self.data = data
        self.update_rate = update_rate
        self.times=times

        self.counter = 0
        self.sim_steps = 0
        self.measured_field = []

    def __call__(self, t: float):
        """
        Get the next value of the magnetic field.

        Parameters
        ----------
        t : float
            Current time.
        """        
        if t == self.times[self.sim_steps]:
            if self.sim_steps % self.update_rate == 0:
                self.measured_field.append(self.data[self.counter])
                self.counter += 1

            self.sim_steps += 1

        return 10 * self.measured_field[-1]

# Run Simulation

In [ ]:
@dataclass
class SimulationParameters:
    """
    Helper class for the simulation parameters.
    """
    length: int
    coupling: list

@dataclass
class SimulationState:
    """ 
    Helper class for the simulation state.
    """
    number_of_spins: int
    quantum_state: list
    spin_list: list
    coupling_list: list

In [ ]:
def get_simulation_state(parameters: SimulationParameters):
    """
    Returns the initial state of the simulation.
    """
    # Get the initial wavefunction
    number_of_spins = parameters.length
    initial_state = [basis(2, 1)] + [basis(2, 0)] * (number_of_spins - 1)

    # Setup operators for individual qubits
    sx_list, sy_list, sz_list = [], [], []
    for i in range(number_of_spins):
        op_list = [qeye(2)] * number_of_spins
        op_list[i] = sigmax()
        sx_list.append(tensor(op_list))
        op_list[i] = sigmay()
        sy_list.append(tensor(op_list))
        op_list[i] = sigmaz()
        sz_list.append(tensor(op_list))

    # Setup the operators for the coupling
    Jx = parameters.coupling * np.ones(number_of_spins)
    Jy = parameters.coupling * np.ones(number_of_spins)
    Jz = parameters.coupling * np.ones(number_of_spins)

    return SimulationState(
        number_of_spins=number_of_spins,
        quantum_state=tensor(initial_state),
        spin_list=[sx_list, sy_list, sz_list],
        coupling_list=[Jx, Jy, Jz],
    )

In [ ]:
def compute_hamiltonian(t, args):
    """
    Compute the Hamiltonian at time t.
    
    Parameters
    ----------
    t : float
        Current time.
    args : dict
        System parameters in the H computation.
    """
    sx_list = args["sx_list"]
    sy_list = args["sy_list"]
    sz_list = args["sz_list"]
    Jx = args["Jx"]
    Jy = args["Jy"]
    Jz = args["Jz"]
    driving_field = args["driving"]

    N = args["N"]

    # Magnetic field terms to top row
    H = 0
    field_values = driving_field(t)

    for i in range(N):
        H -= field_values * sx_list[i]
        H -= field_values * sy_list[i]
        H -= field_values * sz_list[i]


    # Interaction terms
    for n in range(N - 1):
        H += -0.5 * Jx[n] * sx_list[n] * sx_list[n + 1]
        H += -0.5 * Jy[n] * sy_list[n] * sy_list[n + 1]
        H += -0.5 * Jz[n] * sz_list[n] * sz_list[n + 1]

    return H

In [ ]:
apply_frequency = 0.8  # Chnage the field every second
simulation_time = apply_frequency * sampled_data.shape[0] - 1
time_step = 0.1
update_frequency = int(apply_frequency / time_step)
simulation_steps = int(simulation_time / time_step)

times = np.linspace(0, simulation_time, simulation_steps)

simulation_parameters = SimulationParameters(
    length=5,
    coupling=0.1 * np.pi,
)
simulation_state = get_simulation_state(simulation_parameters)

field_generator = BFieldGenerator(
    data=sampled_data,
    update_rate=update_frequency,
    times=times
)

args = {
    "sx_list": simulation_state.spin_list[0],
    "sy_list": simulation_state.spin_list[1],
    "sz_list": simulation_state.spin_list[2],
    "Jx": simulation_state.coupling_list[0],
    "Jy": simulation_state.coupling_list[1],
    "Jz": simulation_state.coupling_list[2],
    "N": simulation_state.number_of_spins,
    "driving": field_generator
}

In [ ]:
results = mesolve(compute_hamiltonian, simulation_state.quantum_state, times, [], [], args)

In [ ]:
measured_field = np.array(field_generator.measured_field)[:]

In [ ]:
b_field = measured_field.reshape(-1, 1)

In [ ]:
plt.plot(b_field)

In [ ]:
measure_states = results.states[update_frequency - 1::update_frequency]

In [ ]:
state_dimension = 1000
states = 3
measurements = []

for _ in range(state_dimension):
    non_gue_matrix = qutip.rand_herm(
        2**states, 
        1.0, 
        dims=[[2] * states, [2] * states]
    )
    measurements.append(non_gue_matrix)

In [ ]:
measure_sites = []
for _ in range(state_dimension):
    measure_sites.append(
        np.random.choice(5, states, replace=False)
    )

In [ ]:
# Compute observables
observations = np.zeros((len(measure_states), state_dimension))
states = measure_states

for t, state in track(enumerate(states)):
    for o, operator in enumerate(measurements):
        measure_state = state.ptrace(measure_sites[o])
        observations[t][o] = qutip.expect(measure_state * measure_state.dag(), operator)

# Fit Model

In [ ]:
class NeuralNetwork(nn.Module):
    
    def __init__(self, state_dimension: int, output_dimension: int):
        """
        Build the network.
        
        Parameters
        ----------
        state_dimension : int
                Dimension of the state representation.
                This is the input to the layer.
        output_dimension : int
                Dimension of the output being predicted.
        """
        super().__init__()

        self.linear_stack = nn.Sequential(
            nn.Linear(state_dimension, 1024),
            nn.ReLU(),
            nn.Linear(1024, output_dimension)
        )
    
    def forward(self, x):
        """
        Forward pass through the network.
        
        As we are doing reservoir computing, this is 
        simply a linear layer.
        """
        # return self.readout_layer(x)
        return self.linear_stack(x)

In [ ]:
class ReservoirDataset(Dataset):
    """
    Custom dataset for the training.
    """
    def __init__(
        self, 
        state_data: np.ndarray, 
        prediction_length: int,
        function_data: np.ndarray,
    ):
        """
        Constructor for the dataset.

        Parameters
        ----------
        state_data : np.ndarray
                State description data.
        function_data : np.ndarray
                Function data being fit.
                This will be the target data.
        prediction_length : int
                How far into the future you will predict.
        """
        self.state_data = torch.Tensor(state_data).to(device)

        self.function_data = torch.Tensor(function_data).to(device)

        self.norm_factor = 1

    def split_data(self, train_size: float):
        """
        Split the data into training and validation sets.
        
        Parameters
        ----------
        train_size : float
                Size of the training set.
        """
        train_size = int(train_size * len(self))
        val_size = len(self) - train_size
        return torch.utils.data.random_split(self, [train_size, val_size])
    
    def __len__(self):
        """
        Length of the dataset.
        """
        return int(
            len(self.function_data)
        )
    
    def __getitem__(self, idx: int):
        """
        Collect an item from the dataset.
        
        Parameters
        ----------
        idx : int
                Index of the state to take.
        """
        state = self.state_data[idx]
        target = self.function_data[idx] / self.norm_factor
        
        return state, target

In [ ]:
def train(dataloader, model, loss_fn, optimizer):
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)
        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            
    return loss

In [ ]:
def test(dataloader, model, loss_fn) -> np.ndarray:
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss = 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
    test_loss /= num_batches
    
    return test_loss

In [ ]:
def train_model(dataset, test_ds, model = None):
    """
    Train a model on the current data.
    """    
    if model is None:
        model = NeuralNetwork(
            state_dimension=state_dimension,
            output_dimension=1
        ).to(device)

        model = model.type(torch.float32)

    # Use MSE loss
    loss_fn = nn.MSELoss()

    # Use ADAM optimizer
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)

    # Create the loader
    loader = DataLoader(dataset, batch_size=300, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=300, shuffle=False)
    
    # Train the network
    epochs = 2000
    train_losses = []
    test_losses = []

    for t in track(range(epochs)):
        loss = train(loader, model, loss_fn, optimizer)
        train_losses.append(loss)
        loss = test(test_loader, model, loss_fn)
        test_losses.append(loss)

    try:
        train_losses = [item.cpu().detach().numpy() for item in train_losses]
    except AttributeError:
        pass
    
    return train_losses, test_losses, model

## Train the model

In [ ]:
prediction_length = 1
train_fraction = 0.6

mix_train_indices = np.random.choice(len(observations) - prediction_length - 1, int(train_fraction * len(observations)), replace=False)
mix_test_indices = np.array([i for i in range(len(observations) - prediction_length - 1) if i not in mix_train_indices])

sequence_train_indices = np.linspace(0, int(train_fraction * len(observations)), int(train_fraction * len(observations)), dtype=int)
sequence_test_indices = np.array([i for i in range(len(observations) - prediction_length - 1) if i not in sequence_train_indices])

train_indices = sequence_train_indices
test_indices = sequence_test_indices

train_dataset = ReservoirDataset(
    state_data=observations[train_indices],
    function_data=b_field[train_indices + prediction_length],
    prediction_length=1
)

test_dataset = ReservoirDataset(
    state_data=observations[test_indices],
    function_data=b_field[test_indices + prediction_length],
    prediction_length=1
)
train_losses, test_losses, model = train_model(train_dataset, test_dataset, model=None)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 5))

ax[0].plot(test_losses)
ax[1].plot(train_losses)

ax[1].set_yscale("log")
ax[0].set_yscale("log")

plt.show()

In [ ]:
test_predictions = []
test_targets = []
for item in test_dataset:
    state, target = item
    test_predictions.append(model(state).cpu().detach().numpy())
    test_targets.append(target.cpu().detach().numpy())

test_predictions = np.array(test_predictions)
test_targets = np.array(test_targets)

train_predictions = []
train_targets = []
for item in train_dataset:
    state, target = item
    train_predictions.append(model(state).cpu().detach().numpy())
    train_targets.append(target.cpu().detach().numpy())

train_predictions = np.array(train_predictions)
train_targets = np.array(train_targets)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(30, 5))

ax[0].plot(test_targets, '.')
ax[0].plot(test_predictions)

ax[1].plot(train_targets)
ax[1].plot(train_predictions)

In [ ]:
full_targets = np.concatenate([train_targets, test_targets])
times = np.linspace(0, simulation_time, full_targets.shape[0])

train_times = times[train_indices]
test_times = times[test_indices]

plt.plot(times, full_targets, label="Baseline")
plt.plot(train_times, train_predictions, '.-', label="Train", mfc="none")
plt.plot(test_times, test_predictions, '.-', label="Test", mfc="none")

plt.xlabel("Time / s")
plt.ylabel("B Field")
plt.legend(loc="upper left")

plt.savefig("mg-equation.pdf")
plt.show()
